# 02. Geração de Pseudo-Labels com DeepForest

Este notebook executa o modelo pré-treinado DeepForest (RetinaNet) sobre as imagens compactadas no arquivo HDF5, converte as caixas para o formato YOLO [class_id, x_center, y_center, w, h] normalizado, e salva as anotações geradas no subgrupo `labels` correspondente.

**Responsável:** Pessoa 3

In [18]:
import h5py
import numpy as np
import pandas as pd

from deepforest import main

## 1. Carregamento do HDF5 e do Modelo DeepForest

In [22]:
HDF5_PATH = "dataset_v1_raw.h5"

hdf5 = h5py.File("dataset_v1_raw.h5", "r")
images = hdf5["images"]

print("Total de imagens:", len(images))

model = main.deepforest()
model.load_model()

print("Modelo carregado com sucesso")

Total de imagens: 0


GPU available: False, used: False
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


Modelo carregado com sucesso


## 2. Predição de Caixas e Normalização YOLO (Pessoa 3)

Conversão de [xmin, ymin, xmax, ymax] em pixels para as coordenadas normalizadas [0.0, 1.0].

In [ ]:
def convert_to_yolo(box, img_w, img_h):
    xmin, ymin, xmax, ymax = box

    x_center = (xmin + xmax) / 2 / img_w
    y_center = (ymin + ymax) / 2 / img_h
    width = (xmax - xmin) / img_w
    height = (ymax - ymin) / img_h

    return [x_center, y_center, width, height]


pseudo_labels = []

for i in range(len(images)):

    img = images[i]

    if img.dtype != np.uint8:
        img = img.astype(np.uint8)

    preds = model.predict_image(image=img)

    h, w = img.shape[:2]

    if preds is not None and len(preds) > 0:

        for _, row in preds.iterrows():

            box = [
                row["xmin"],
                row["ymin"],
                row["xmax"],
                row["ymax"]
            ]

            yolo_box = convert_to_yolo(box, w, h)

            pseudo_labels.append({
                "image_id": i,
                "class": 0,
                "bbox": yolo_box,
                "score": row["score"]
            })

print("Total de pseudo-labels:", len(pseudo_labels))

## 3. Gravação das Pseudo-Labels no HDF5

In [ ]:
if "labels" in hdf5:
    del hdf5["labels"]

label_group = hdf5.create_group("labels")

image_ids = np.array([p["image_id"] for p in pseudo_labels])
classes = np.array([p["class"] for p in pseudo_labels])
bboxes = np.array([p["bbox"] for p in pseudo_labels])
scores = np.array([p["score"] for p in pseudo_labels])

label_group.create_dataset("image_id", data=image_ids)
label_group.create_dataset("class", data=classes)
label_group.create_dataset("bbox", data=bboxes)
label_group.create_dataset("score", data=scores)

hdf5.close()

print("Pseudo-labels salvas com sucesso no HDF5")